# Threshold Analysis for the inference model

In [1]:
# imports
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, f1_score

# helpers
sys.path.insert(0, "../../")

from _settings import LABEL_COLS
from dataviz_helper import plot_coverage_precision_curve
from models_helper import softvote_pred_and_confidence, sweep_thresholds, best_conf_threshold_at_min_coverage
from ensemble_helper import EnsemblePredictor

In [2]:
# define directories and constants
DATA_DIR = Path("../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes")
OUTPUT_PATH = Path("threshold_analysis_results.json")
THRESHOLDS = np.arange(0.0, 1.0, 0.05)

## 1. Soft-vote with confidence on val and test

In [3]:
# load val and test set with rare classes removed and extract labels
df_val  = pd.read_parquet(DATA_DIR / "dataset_validation_rare_classes_removed.parquet")
df_test = pd.read_parquet(DATA_DIR / "dataset_test_rare_classes_removed.parquet")
y_val   = df_val[LABEL_COLS].reset_index(drop=True)
y_test  = df_test[LABEL_COLS].reset_index(drop=True)

# run ensemble predictor on val and test set to get proba predictions
predictor   = EnsemblePredictor()
probas_val  = predictor.predict_proba(df_val)
probas_test = predictor.predict_proba(df_test)

# compute soft-vote ensemble predictions and their confidence (max proba) for each label
sv_pred, sv_conf = softvote_pred_and_confidence(probas_val,  LABEL_COLS)
sv_pred_test, sv_conf_test = softvote_pred_and_confidence(probas_test, LABEL_COLS)

# statistics on confidence values per label to get a sense of their distribution
pd.DataFrame({l: pd.Series(sv_conf[l]).describe() for l in LABEL_COLS}).round(4)

xgboost: 3.2.0 imported first
rf - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_rf_model_0.7143.pkl
xgboost - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_xgboost_model_0.7276.pkl
nn: 2.11.0 imported first
model xgboost - demo: 8,458 rows, features: 35
model rf - demo: 8,458 rows, features: 37
model xgboost wrote demo preds + probas
model rf wrote demo preds + probas
nn - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_nn_model_0.6784.pkl
nn - moved to mps
model nn - demo: 8,458 rows, features: 73
model nn wrote demo preds + probas
nn: 2.11.0 imported first
xgboost: 3.2.0 imported first
rf - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_rf_model_0.7143.pkl
xgboost - loading /User

,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing
count,8458.0000,8458.0000,8458.0000,8458.0000
mean,0.8750,0.6636,0.8463,0.8373
std,0.1920,0.1919,0.1498,0.1582
min,0.1961,0.2432,0.3534,0.3716
25%,0.8428,0.5210,0.7460,0.7094
50%,0.9831,0.6436,0.8970,0.8848
75%,0.9966,0.7940,0.9754,0.9842
max,0.9999,0.9980,0.9999,0.9999


## 2. Threshold Sweep for trade-off analysis

In [4]:
# get coverage, accuracy and f1_macro for each label and each threshold
thr_df = sweep_thresholds(y_val, sv_pred, sv_conf, LABEL_COLS, THRESHOLDS)
thr_df

,label,conf_threshold,coverage,n_covered,accuracy,precision,f1_macro,mcc
0,label_ifc_entity,0.00,1.0000,8458,0.8415,0.8447,0.6955,0.7712
1,label_ifc_entity,0.05,1.0000,8458,0.8415,0.8447,0.6955,0.7712
2,label_ifc_entity,0.10,1.0000,8458,0.8415,0.8447,0.6955,0.7712
3,label_ifc_entity,0.15,1.0000,8458,0.8415,0.8447,0.6955,0.7712
4,label_ifc_entity,0.20,0.9999,8457,0.8416,0.8453,0.6957,0.7713
...,...,...,...,...,...,...,...,...
75,label_load_bearing,0.75,0.7022,5939,0.9451,0.9477,0.9429,0.9180
76,label_load_bearing,0.80,0.6302,5330,0.9595,0.9611,0.9554,0.9390
77,label_load_bearing,0.85,0.5549,4693,0.9717,0.9730,0.9657,0.9568
78,label_load_bearing,0.90,0.4790,4051,0.9899,0.9901,0.9862,0.9841


In [5]:
# plot each label for each important metric
for metric in ["coverage", "precision", "f1_macro"]:
    print(f"\n{metric}:")
    display(thr_df.pivot(index="conf_threshold", columns="label", values=metric))


coverage:


label,label_ifc_entity,label_is_external,label_load_bearing,label_predefined_type
conf_threshold,,,,
0.00,1.0000,1.0000,1.0000,1.0000
0.05,1.0000,1.0000,1.0000,1.0000
0.10,1.0000,1.0000,1.0000,1.0000
0.15,1.0000,1.0000,1.0000,1.0000
0.20,0.9999,1.0000,1.0000,1.0000
0.25,0.9987,1.0000,1.0000,0.9991
0.30,0.9934,1.0000,1.0000,0.9873
0.35,0.9821,1.0000,1.0000,0.9689
0.40,0.9602,0.9989,0.9982,0.9259



precision:


label,label_ifc_entity,label_is_external,label_load_bearing,label_predefined_type
conf_threshold,,,,
0.00,0.8447,0.8787,0.8257,0.7887
0.05,0.8447,0.8787,0.8257,0.7887
0.10,0.8447,0.8787,0.8257,0.7887
0.15,0.8447,0.8787,0.8257,0.7887
0.20,0.8453,0.8787,0.8257,0.7887
0.25,0.8505,0.8787,0.8257,0.7888
0.30,0.8519,0.8787,0.8257,0.7915
0.35,0.8569,0.8787,0.8257,0.7969
0.40,0.8624,0.8786,0.8262,0.8063



f1_macro:


label,label_ifc_entity,label_is_external,label_load_bearing,label_predefined_type
conf_threshold,,,,
0.00,0.6955,0.9011,0.8196,0.5150
0.05,0.6955,0.9011,0.8196,0.5150
0.10,0.6955,0.9011,0.8196,0.5150
0.15,0.6955,0.9011,0.8196,0.5150
0.20,0.6957,0.9011,0.8196,0.5150
0.25,0.6981,0.9011,0.8196,0.5153
0.30,0.7018,0.9011,0.8196,0.5186
0.35,0.7144,0.9011,0.8196,0.5239
0.40,0.7148,0.9010,0.8205,0.5346


## 3. Coverage vs. Precision Curve

In [6]:
plot_coverage_precision_curve(thr_df, LABEL_COLS, title="", save="coverage_precision_curve", chapter="implementation")

figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/4 implementation/img/coverage_precision_curve.svg


## 4. Check which confidence threshold would give the best Trade-off

In [7]:
# check which confidence threshold would give the best accuracy while still keeping at least 80% or 50% of elements (rows with confidence above the threshold)
print("Minimum coverage of 70% of elements:")
display(best_conf_threshold_at_min_coverage(thr_df, 0.70, LABEL_COLS))

Minimum coverage of 70% of elements:


,conf_threshold,coverage,precision,f1_macro
label,,,,
label_ifc_entity,0.85,0.7436,0.9177,0.8647
label_predefined_type,0.50,0.7865,0.8508,0.5608
label_is_external,0.75,0.7449,0.9271,0.9341
label_load_bearing,0.75,0.7022,0.9477,0.9429


## 5. Verification on Test Set with Fixed confidence thresholds

In [8]:
# add confidence thresholds to settings file for later use on test set
from _settings import CONFIDENCE_THRESHOLDS

print("\nCurrent confidence thresholds:")
print(CONFIDENCE_THRESHOLDS)


Current confidence thresholds:
{'label_ifc_entity': 0.85, 'label_predefined_type': 0.5, 'label_is_external': 0.75, 'label_load_bearing': 0.75}


In [9]:
# check the performance of the currently chosen confidence thresholds on the test set (picked based on val set analysis above)
unique_thresholds = sorted({round(t, 2) for t in CONFIDENCE_THRESHOLDS.values()})
sweep_test = sweep_thresholds(y_test, sv_pred_test, sv_conf_test, LABEL_COLS, unique_thresholds)

test_verification = (
    pd.concat([
        sweep_test[(sweep_test["label"] == label) & (sweep_test["conf_threshold"] == round(CONFIDENCE_THRESHOLDS[label], 2))]
        for label in LABEL_COLS
    ])
    .assign(tau=lambda d: d["conf_threshold"], n_total=len(y_test))
    .drop(columns=["conf_threshold"])
    .set_index("label")
)
test_verification


,coverage,n_covered,accuracy,precision,f1_macro,mcc,tau,n_total
label,,,,,,,,
label_ifc_entity,0.7661,2719,0.9926,0.9906,0.7625,0.9875,0.85,3549
label_predefined_type,0.7647,2714,0.8106,0.8205,0.5519,0.7696,0.50,3549
label_is_external,0.7095,2518,0.9392,0.9456,0.9417,0.9122,0.75,3549
label_load_bearing,0.7092,2517,0.9098,0.9226,0.8924,0.8496,0.75,3549


## 6. Save all results as JSON for the report

In [10]:
# save results as json
results = {
    "sweep_val": thr_df.to_dict(orient="records"),
    "chosen_tau": CONFIDENCE_THRESHOLDS,
    "test_verification": test_verification.reset_index().to_dict(orient="records"),
}

OUTPUT_PATH.write_text(json.dumps(results, indent=2, default=lambda v: None if pd.isna(v) else v))
print(f"saved to: {OUTPUT_PATH.resolve()}")

saved to: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/5_evaluation/5_5_threshold_inference_analysis/threshold_analysis_results.json
